In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Memory experiment analysis: min-distance + ROC curves.
Refactored into a single organized script. 
Modify only `run_experiment_at_noise` to explore new dynamics.
"""

# ===================== Imports =====================
import sys, os, glob, json, math, datetime
from collections import defaultdict

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.linear_model import LinearRegression

# project-specific paths
sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')
sys.path.append('../utls/')
sys.path.append('../src/model/')
sys.path.append("/om2/user/bjmedina/auditory-memory/memory/")

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

import DistanceMemoryModel
import encoders
from utls.plotting import ensure_dir
from utls.loading import load_results_with_exclusion_2, move_sequences_to_used
from utls.runners import run_experiment_scores
from utls.analysis_helpers import rocs_across_noise, convert_human_to_model_struct, compute_scaling_vs_human, convert_human_to_model_struct
from utls.analysis_helpers import auroc_to_dprime, compute_model_dprime_curve
from utls.analysis_helpers import roc_for_isi, auroc_to_dprime
from utls.plotting import plot_across_noise, plot_noise_overlays
from utls.io_utils import make_model_save_dir, save_all_figures, save_single_figure, save_runs_summary

import os, json
import numpy as np
from scipy.stats import norm
from sklearn.utils import resample
from utls.roc_utils import roc_from_arrays  # if separate, or inline

In [ ]:
files = glob.glob("/mindhive/mcdermott/www/bjmedina/experiments/mem_exp_v02/results/*V15.csv")
exps_, seqs_, fnames_ = load_results_with_exclusion_2(
    "/mindhive/mcdermott/www/bjmedina/experiments/mem_exp_v02/results/V15",
    min_dprime=2, min_trials=120, skip_len60=True, verbose=False, return_skipped=False
)

base_path_ = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_v15/sequences/"

# Build experiment_list
experiment_list_ = []
for seq_ in seqs_:
    with open(base_path_ + seq_, 'r') as f:
        data = json.load(f)
    stim_paths = ["/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_v15/" + s 
                  for s in data['filenames_order']]

    experiment_list_.append(stim_paths)

In [ ]:
save_base="/om2/user/bjmedina/auditory-memory/memory/figures/model-behavior-step-functions/"
t_results = glob.glob("/mindhive/mcdermott/www/bjmedina/experiments/mem_exp_v02/results/*V15.csv")

PC_DIMS = 256
DEVICE = 'cuda'
NV_DEFAULT = 0.1

# Encode clean reps
pc_texture_model = encoders.AudioTextureEncoderPCA(
    statistics_dict=statistics_set.statistics,
    pc_dims=PC_DIMS, model_params=model_params,
    sr=20000, rms_level=0.05, duration=2.0, device=DEVICE
)


all_files_ = sorted({fn for seq in experiment_list_ for fn in seq})

tmp = DistanceMemoryModel.DistanceMemoryModel(pc_texture_model, NV_DEFAULT, criterion=0.0, device=DEVICE)
tmp._fill_memory_bank(all_files_)
with torch.no_grad():
    X0_ = torch.stack([rep.detach().clone().view(-1) for rep in tmp.memory_bank], dim=0).to(DEVICE)
    
#name_to_idx = {fn: i for i, fn in enumerate(all_files)}
name_to_idx_ = {fn: i for i, fn in enumerate(all_files_)}

human_runs = []
for result in t_results:
    df = pd.read_csv(result)
    main_exp = df[df['stim_type'] == 'main']
    if len(main_exp) < 256:
        continue
    human_runs.append(convert_human_to_model_struct(main_exp))

# Compute average human d′ vs ISI

isis_human = [0, 1, 2, 3, 4, 8, 16, 32, 64]
dprimes_human = []
for k in isis_human:
    aucs = []
    for run in human_runs:
        res = roc_for_isi(run, k)
        if res is not None:
            _, _, auc = res
            aucs.append(auroc_to_dprime(auc))
    dprimes_human.append(np.nanmean(aucs))
    
human_curve = np.array(dprimes_human, dtype=float)
human_curve = human_curve[~np.isnan(human_curve)]

In [ ]:

print(save_base)

In [ ]:
def compute_snr_for_repeats(X0, name_to_idx, experiment_list,
                            sigma0, noise_mode="diffuse", rate=None, runs=100):
    """
    Compute per-ISI representational SNR (in dB) for repeated sounds,
    where memories drift continuously between presentations.
    
    Each memory gets noisy updates at each time step, and when a repeat occurs,
    SNR is computed as 20*log10(||clean|| / ||noisy - clean||).
    
    Args:
        X0 : torch.Tensor (N x D)
        name_to_idx : dict[str → int]
        experiment_list : list[list[str]]
        sigma0 : float, base noise level
        noise_mode : str, one of {"diffuse","decay","power-law","power-decay","exp-law","exp-decay","const"}
        rate : float, optional exponent/rate for decay/law schedules
        runs : int, number of Monte Carlo noise runs (averaging stability)
    
    Returns:
        dict: {ISI (int): [snr_dB_values]}
    """
    dim_std = X0.std(0, unbiased=True)
    scaled_std = dim_std / dim_std.max()
    snr_by_isi = defaultdict(list)

    for _ in range(runs):
        for seq in experiment_list:
            if not seq:
                continue

            seq_idx = [name_to_idx[fn] for fn in seq]
            memory_bank, last_seen = {}, {}

            for t, idx_incoming in enumerate(seq_idx, start=1):
                clean = X0[idx_incoming].detach().clone()

                # ---- continuous drift of all existing memories ----
                for mem in memory_bank.values():
                    age = t - mem["t_inserted"]
                    if age <= 0:
                        continue

                    if noise_mode == "diffuse":
                        std = sigma0 * np.sqrt(age)
                    elif noise_mode == "power-decay":
                        std = sigma0 / (age ** (rate if rate is not None else 0.5))
                    elif noise_mode == "power-law":
                        std = sigma0 * (age ** (rate if rate is not None else 0.5))
                    elif noise_mode == "exp-decay":
                        std = sigma0 / np.exp(age * (rate if rate is not None else 1.0))
                    elif noise_mode == "exp-law":
                        std = sigma0 * np.exp(age * (rate if rate is not None else 1.0))
                    elif noise_mode == "const":
                        std = sigma0
                    elif noise_mode == "linear-step":
                        # Rapid early growth, then plateau
                        t_step = 5
                        if age < t_step:
                            std = sigma0 * (1 + rate * age)
                        else:
                            std = 0#sigma0 * (1 + rate * t_step)
                    elif noise_mode == "const-step":
                        # Rapid early growth, then plateau
                        t_step = 5
                        if age < t_step:
                            std = sigma0 
                        else:
                            std = 0#sigma0 * (1 + rate * t_step)
                    elif noise_mode == "exp-step":
                        # Rapid early growth, then plateau
                        t_step = 5
                        if age < t_step:
                            std = sigma0 * np.exp(age * (rate if rate is not None else 1.0))
                        else:
                            std = 0
                    elif noise_mode == "linear-step-v2":
                        # Rapid early growth, then plateau
                        t_step = 5
                        if age < t_step:
                            std = sigma0 * (1 + rate * age)
                        else:
                            std = sigma0 * (1 + rate * t_step)
                    elif noise_mode == "exp-step-v2":
                        # Rapid early growth, then plateau
                        t_step = 5
                        if age < t_step:
                            std = sigma0 * np.exp(age * (rate if rate is not None else 1.0))
                        else:
                            std = sigma0 * np.exp(t_step * (rate if rate is not None else 1.0))
                    std = max(std, 1e-6)

                    noise = torch.randn_like(mem["mu"]) * (std * scaled_std)
                    mem["mu"] = mem["mu"] + noise
                    mem["age"] = age

                # ---- if repeated sound: measure SNR ----
                if idx_incoming in last_seen:
                    isi = t - last_seen[idx_incoming]
                    mem = memory_bank[idx_incoming]
                    noisy = mem["mu"]

                    snr_db = 20 * np.log10(
                        torch.norm(clean).item() / (torch.norm(noisy - clean).item() + 1e-12)
                    )
                    snr_by_isi[isi].append(snr_db)

                # ---- update or insert current sound ----
                memory_bank[idx_incoming] = {"mu": clean.clone(), "t_inserted": t}
                last_seen[idx_incoming] = t

    return snr_by_isi

noise_mode="const"

rates = [0.01, 0.1, 0.3, 0.5]
NOISE_LEVELS = np.geomspace(0.01, 5, 6)
ncols = 2
nrows = int(np.ceil(len(rates)/ncols))

# Precompute all results so we can find global limits
all_snr_means = []
all_isi_vals = []

results_cache = {}

for rate in rates:
    for nv in NOISE_LEVELS:
        snr_results = compute_snr_for_repeats(
            X0_, name_to_idx_, experiment_list_[:20],
            sigma0=nv, noise_mode=noise_mode,
            runs=1, rate=rate
        )
        if not snr_results:
            continue

        isi_vals = sorted(snr_results.keys())
        snr_means = [np.mean(snr_results[k]) for k in isi_vals]
        snr_sems  = [np.std(snr_results[k]) / np.sqrt(len(snr_results[k])) for k in isi_vals]

        results_cache[(rate, nv)] = (isi_vals, snr_means, snr_sems)
        all_snr_means.extend(snr_means)
        all_isi_vals.extend(isi_vals)

# Compute global axis limits
x_min, x_max = min(all_isi_vals), max(all_isi_vals)
y_min, y_max = np.percentile(all_snr_means, [1, 99])  # robust limits

# Make plots
fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 4*nrows), squeeze=False)

for i, rate in enumerate(rates):
    ax = axes[i // ncols, i % ncols]

    for nv in NOISE_LEVELS:
        key = (rate, nv)
        if key not in results_cache:
            continue
        isi_vals, snr_means, snr_sems = results_cache[key]
        ax.errorbar(
            isi_vals, snr_means, yerr=snr_sems,
            fmt='o-', capsize=3, alpha=0.6, label=f"σ₀={nv:g}"
        )

    ax.set_xlim(x_min-1, x_max+1)
    ax.set_ylim(y_min-2, y_max+2)
    ax.set_xlabel("ISI (memory age)")
    ax.set_ylabel("Mean representational SNR (dB)")
    ax.set_title(f"Rate = {rate}")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)

plt.suptitle(f"SNR across ISI for different σ₀ values | {noise_mode}", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(save_base + f"snr_by_isi-{noise_mode}.png", dpi=300)
plt.show()
plt.close()

In [ ]:
noise_mode="linear-step"
rates = [0.01, 0.1, 0.3, 0.5]
NOISE_LEVELS = np.geomspace(0.01, 5, 6)
ncols = 2
nrows = int(np.ceil(len(rates)/ncols))

# Precompute all results so we can find global limits
all_snr_means = []
all_isi_vals = []

results_cache = {}

for rate in rates:
    for nv in NOISE_LEVELS:
        snr_results = compute_snr_for_repeats(
            X0_, name_to_idx_, experiment_list_[:15],
            sigma0=nv, noise_mode=noise_mode,
            runs=1, rate=rate
        )
        if not snr_results:
            continue

        isi_vals = sorted(snr_results.keys())
        snr_means = [np.mean(snr_results[k]) for k in isi_vals]
        snr_sems  = [np.std(snr_results[k]) / np.sqrt(len(snr_results[k])) for k in isi_vals]

        results_cache[(rate, nv)] = (isi_vals, snr_means, snr_sems)
        all_snr_means.extend(snr_means)
        all_isi_vals.extend(isi_vals)

# Compute global axis limits
x_min, x_max = min(all_isi_vals), max(all_isi_vals)
y_min, y_max = np.percentile(all_snr_means, [1, 99])  # robust limits

# Make plots
fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 4*nrows), squeeze=False)

for i, rate in enumerate(rates):
    ax = axes[i // ncols, i % ncols]

    for nv in NOISE_LEVELS:
        key = (rate, nv)
        if key not in results_cache:
            continue
        isi_vals, snr_means, snr_sems = results_cache[key]
        ax.errorbar(
            isi_vals, snr_means, yerr=snr_sems,
            fmt='o-', capsize=3, alpha=0.6, label=f"σ₀={nv:g}"
        )

    ax.set_xlim(x_min-1, x_max+1)
    ax.set_ylim(y_min-2, y_max+2)
    ax.set_xlabel("ISI (memory age)")
    ax.set_ylabel("Mean representational SNR (dB)")
    ax.set_title(f"Rate = {rate}")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)

plt.suptitle(f"SNR across ISI for different σ₀ values | {noise_mode}", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(save_base + f"snr_by_isi-{noise_mode}.png", dpi=300)
plt.show()
plt.close()

In [ ]:
noise_mode="exp-step"

rates = [0.01, 0.1, 0.3, 0.5]
NOISE_LEVELS = np.geomspace(0.01, 5, 6)
ncols = 2
nrows = int(np.ceil(len(rates)/ncols))

# Precompute all results so we can find global limits
all_snr_means = []
all_isi_vals = []

results_cache = {}

for rate in rates:
    for nv in NOISE_LEVELS:
        snr_results = compute_snr_for_repeats(
            X0_, name_to_idx_, experiment_list_[:15],
            sigma0=nv, noise_mode=noise_mode,
            runs=1, rate=rate
        )
        if not snr_results:
            continue

        isi_vals = sorted(snr_results.keys())
        snr_means = [np.mean(snr_results[k]) for k in isi_vals]
        snr_sems  = [np.std(snr_results[k]) / np.sqrt(len(snr_results[k])) for k in isi_vals]

        results_cache[(rate, nv)] = (isi_vals, snr_means, snr_sems)
        all_snr_means.extend(snr_means)
        all_isi_vals.extend(isi_vals)

# Compute global axis limits
x_min, x_max = min(all_isi_vals), max(all_isi_vals)
y_min, y_max = np.percentile(all_snr_means, [1, 99])  # robust limits

# Make plots
fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 4*nrows), squeeze=False)

for i, rate in enumerate(rates):
    ax = axes[i // ncols, i % ncols]

    for nv in NOISE_LEVELS:
        key = (rate, nv)
        if key not in results_cache:
            continue
        isi_vals, snr_means, snr_sems = results_cache[key]
        ax.errorbar(
            isi_vals, snr_means, yerr=snr_sems,
            fmt='o-', capsize=3, alpha=0.6, label=f"σ₀={nv:g}"
        )

    ax.set_xlim(x_min-1, x_max+1)
    ax.set_ylim(y_min-2, y_max+2)
    ax.set_xlabel("ISI (memory age)")
    ax.set_ylabel("Mean representational SNR (dB)")
    ax.set_title(f"Rate = {rate}")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)

plt.suptitle(f"SNR across ISI for different σ₀ values | {noise_mode}", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(save_base + f"snr_by_isi-{noise_mode}.png", dpi=300)
plt.show()
plt.close()

In [ ]:
noise_mode="linear-step-v2"

rates = [0.01, 0.1, 0.3, 0.5]
NOISE_LEVELS = np.geomspace(0.01, 5, 6)
ncols = 2
nrows = int(np.ceil(len(rates)/ncols))

# Precompute all results so we can find global limits
all_snr_means = []
all_isi_vals = []

results_cache = {}

for rate in rates:
    for nv in NOISE_LEVELS:
        snr_results = compute_snr_for_repeats(
            X0_, name_to_idx_, experiment_list_[:15],
            sigma0=nv, noise_mode=noise_mode,
            runs=1, rate=rate
        )
        if not snr_results:
            continue

        isi_vals = sorted(snr_results.keys())
        snr_means = [np.mean(snr_results[k]) for k in isi_vals]
        snr_sems  = [np.std(snr_results[k]) / np.sqrt(len(snr_results[k])) for k in isi_vals]

        results_cache[(rate, nv)] = (isi_vals, snr_means, snr_sems)
        all_snr_means.extend(snr_means)
        all_isi_vals.extend(isi_vals)

# Compute global axis limits
x_min, x_max = min(all_isi_vals), max(all_isi_vals)
y_min, y_max = np.percentile(all_snr_means, [1, 99])  # robust limits

# Make plots
fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 4*nrows), squeeze=False)

for i, rate in enumerate(rates):
    ax = axes[i // ncols, i % ncols]

    for nv in NOISE_LEVELS:
        key = (rate, nv)
        if key not in results_cache:
            continue
        isi_vals, snr_means, snr_sems = results_cache[key]
        ax.errorbar(
            isi_vals, snr_means, yerr=snr_sems,
            fmt='o-', capsize=3, alpha=0.6, label=f"σ₀={nv:g}"
        )

    ax.set_xlim(x_min-1, x_max+1)
    ax.set_ylim(y_min-2, y_max+2)
    ax.set_xlabel("ISI (memory age)")
    ax.set_ylabel("Mean representational SNR (dB)")
    ax.set_title(f"Rate = {rate}")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)

plt.suptitle(f"SNR across ISI for different σ₀ values | {noise_mode}", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(save_base + f"snr_by_isi-{noise_mode}.png", dpi=300)
plt.show()
plt.close()

In [ ]:
noise_mode="exp-step-v2"

rates = [0.01, 0.1, 0.3, 0.5]
NOISE_LEVELS = np.geomspace(0.01, 5, 6)
ncols = 2
nrows = int(np.ceil(len(rates)/ncols))

# Precompute all results so we can find global limits
all_snr_means = []
all_isi_vals = []

results_cache = {}

for rate in rates:
    for nv in NOISE_LEVELS:
        snr_results = compute_snr_for_repeats(
            X0_, name_to_idx_, experiment_list_[:15],
            sigma0=nv, noise_mode=noise_mode,
            runs=1, rate=rate
        )
        if not snr_results:
            continue

        isi_vals = sorted(snr_results.keys())
        snr_means = [np.mean(snr_results[k]) for k in isi_vals]
        snr_sems  = [np.std(snr_results[k]) / np.sqrt(len(snr_results[k])) for k in isi_vals]

        results_cache[(rate, nv)] = (isi_vals, snr_means, snr_sems)
        all_snr_means.extend(snr_means)
        all_isi_vals.extend(isi_vals)

# Compute global axis limits
x_min, x_max = min(all_isi_vals), max(all_isi_vals)
y_min, y_max = np.percentile(all_snr_means, [1, 99])  # robust limits

# Make plots
fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 4*nrows), squeeze=False)

for i, rate in enumerate(rates):
    ax = axes[i // ncols, i % ncols]

    for nv in NOISE_LEVELS:
        key = (rate, nv)
        if key not in results_cache:
            continue
        isi_vals, snr_means, snr_sems = results_cache[key]
        ax.errorbar(
            isi_vals, snr_means, yerr=snr_sems,
            fmt='o-', capsize=3, alpha=0.6, label=f"σ₀={nv:g}"
        )

    ax.set_xlim(x_min-1, x_max+1)
    ax.set_ylim(y_min-2, y_max+2)
    ax.set_xlabel("ISI (memory age)")
    ax.set_ylabel("Mean representational SNR (dB)")
    ax.set_title(f"Rate = {rate}")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)

plt.suptitle(f"SNR across ISI for different σ₀ values | {noise_mode}", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(save_base + f"snr_by_isi-{noise_mode}.png", dpi=300)
plt.show()
plt.close()

In [ ]:
scaling_results = compute_scaling_vs_human(runs, NOISE_LEVELS, human_curve)
r2s = [scaling_results[nv]['r2'] for nv in NOISE_LEVELS]

plt.figure(figsize=(7,5))
plt.plot(NOISE_LEVELS, r2s, 'o--', color='orange', label='$R^2$ fit quality')

plt.ylim([min(np.min(r2s), 0)-0.01,1])
plt.xscale('log')
plt.xlabel("Noise level (σ₀)")
plt.ylabel("$R^2$ (fit quality)")
plt.title("Model–Human Universality Fit Across Noise Levels")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(f"{save_dir}/Universality_fit.png")
plt.show()
plt.close()
#compute_model_dprime_curve(runs, (1,2,3,4,9,17,33,65))

In [ ]:
# Then overlay all ROCs across noise levels:
plot_noise_overlays(curves, save_dir=save_dir)


plot_across_noise(NOISE_LEVELS, runs, bins=60, xmax_percentile=99.5, isis=(1,2,3,4,9,17,33,65),
                  model_info=model_info,
                  save_base=save_base,
                  dprimes_human=dprimes_human,
                  isi_human=(0, 1, 2, 3, 4, 8, 16, 32, 64))

save_runs_summary(runs, model_info, save_dir)

In [ ]:
y_ref    = human_curve
isi_ref = [0, 1, 2, 3, 4, 8, 16, 32, 64]

best_idx = int(np.nanargmax(r2s))
best_nv  = NOISE_LEVELS[best_idx]
best_fit = scaling_results[best_nv]
print(f"Best σ₀ = {best_nv:.3g}  (α={best_fit['alpha']:.3f}, β={best_fit['beta']:.3f}, R²={best_fit['r2']:.3f})")

# Compute model curve
_, y_model = compute_model_dprime_curve(runs[best_nv])
y_scaled = best_fit['alpha'] * y_model + best_fit['beta'] 
print(y_model)# scale model to human space

plt.figure(figsize=(7,5))
plt.errorbar(isi_ref, y_ref, fmt='o-', color='black', label='Human data')
plt.plot(isi_ref, y_model, 's--', color='gray', alpha=0.5, label=f'Model (σ₀={best_nv:g})')
plt.plot(isi_ref, y_scaled, 'o--', color='orange', label=f'Model scaled → Human (α={best_fit["alpha"]:.2f}, β={best_fit["beta"]:.2f})')

plt.xlabel("ISI")
plt.ylabel("d′ (z(AUROC))")
plt.title(f"Best universality fit (σ₀={best_nv:g}, R²={best_fit['r2']:.2f})")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(f"{save_dir}/best_universality_fit.png")
plt.show()
plt.close()

In [ ]:
import pickle, glob, os

def load_all_runs(base_dir):
    all_runs = []
    for pkl_path in glob.glob(os.path.join(base_dir, "**/runs.pkl"), recursive=True):
        info_path = os.path.join(os.path.dirname(pkl_path), "info.json")
        with open(pkl_path, "rb") as f:
            runs = pickle.load(f)
        info = {}
        if os.path.exists(info_path):
            with open(info_path) as f:
                info = json.load(f)
        all_runs.append({"runs": runs, "info": info})
    return all_runs

all_data = load_all_runs("/om2/user/bjmedina/auditory-memory/memory/figures/model-behavior_test/")
print(f"Loaded {len(all_data)} models.")

In [ ]:
# --- Group data ---
grouped = {}
for entry in all_data:
    info = entry["info"]["model_info"]
    metric = info.get("metric", "unknown")
    rate = float(info.get("rate", 0.0))
    grouped.setdefault(metric, {})[rate] = entry

metrics = sorted(grouped.keys())
rates = sorted({r for g in grouped.values() for r in g.keys()})
nrows, ncols = len(rates), len(metrics)

# --- Multi-panel d′ vs ISI across noise levels ---
fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 4*nrows), squeeze=False)
for i, rate in enumerate(rates):
    for j, metric in enumerate(metrics):
        ax = axes[i, j]
        entry = grouped.get(metric, {}).get(rate, None)
        if entry is None:
            ax.axis("off")
            continue

        runs = entry["runs"]
        noise_levels = sorted(runs.keys())

        for nv in noise_levels:
            run_data = runs[nv]
            isis, dprimes = compute_dprime_curve(run_data)
            ax.plot(isis, dprimes, marker='o', alpha=0.5, label=f"σ₀={nv:g}")

        ax.set_title(f"{metric} | rate={rate}")
        ax.set_xlabel("ISI")
        ax.set_ylabel("d′")
        ax.set_ylim(0, 7)
        ax.grid(True, alpha=0.3)
        if i == 0 and j == ncols - 1:
            ax.legend(fontsize=6)
plt.suptitle("d′ vs ISI across noise levels (Model × Rate grid)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
t_results = glob.glob("/mindhive/mcdermott/www/bjmedina/experiments/mem_exp_v02/results/*V15.csv")

from collections import defaultdict
import numpy as np

def convert_human_to_model_struct(main_exp):
    """
    Convert a participant dataframe into model-compatible format.
    
    main_exp : pandas.DataFrame
        Must include columns ['stimulus', 'repeat', 'isi', 'response'].
    
    Returns a dict matching the model-run output format.
    """
    # Convert types
    is_repeat = np.array(main_exp['repeat'] == 'true', dtype=bool)
    isi = np.array(main_exp['isi'], dtype=float)
    response = np.array(main_exp['response'], dtype=int)

    # --- Separate hit / false alarm "scores" ---
    # We'll use responses as "scores" (higher = more confident yes)
    hits = response[is_repeat]
    fas  = response[~is_repeat]

    # --- Build isi_hit_dists like the model structure ---
    isi_hit_dists = defaultdict(list)
    for i, (rep, isi_val, resp) in enumerate(zip(is_repeat, isi, response)):
        if rep and isi_val > -1:  # only valid repeats
            isi_hit_dists[int(isi_val)].append((float(resp), i))

    # --- Optional: FA-by-time list ---
    T_max = len(main_exp)
    fa_by_t = [[] for _ in range(T_max)]
    for t, (rep, resp) in enumerate(zip(is_repeat, response)):
        if not rep:
            fa_by_t[t].append(float(resp))

    return {
        "hits": np.asarray(hits, float),
        "fas": np.asarray(fas, float),
        "isi_hit_dists": isi_hit_dists,
        "fa_by_t": fa_by_t,
        "T_max": T_max,
        "score_type": "likelihood",
        "noise_mode": "human",
    }

human_runs = []
for result in t_results:
    df = pd.read_csv(result)
    main_exp = df[df['stim_type'] == 'main']
    if len(main_exp) < 256:
        continue
    human_runs.append(convert_human_to_model_struct(main_exp))

In [ ]:
from utls.analysis_helpers import roc_for_isi, auroc_to_dprime

# Compute average human d′ vs ISI
isis = [0, 1, 2, 3, 4, 8, 16, 32, 64]
dprimes_human = []

for k in isis:
    aucs = []
    for run in human_runs:
        res = roc_for_isi(run, k)
        if res is not None:
            _, _, auc = res
            aucs.append(auroc_to_dprime(auc))
    dprimes_human.append(np.nanmean(aucs))


# compare this "dprimes_human" to "model dprimes"

In [ ]:
NOISE_LEVELS = np.linspace(0.1, 1, 4)

curves, runs = rocs_across_noise(
    NOISE_LEVELS,
    runner=run_experiment_scores,
    X0=X0_, name_to_idx=name_to_idx_, experiment_list=experiment_list_[:10],
    metric="loglikelihood",
    noise_mode="diffuse"#   # or "loglikelihood"
)
 
plot_noise_overlays(curves)
# for nv in NOISE_LEVELS:
#     if nv in runs:
#         plot_isi_and_half_for_noise(nv, runs[nv], isis=(1,17))
#         plot_histograms_for_noise(nv, runs[nv], bins=60)

plot_across_noise(NOISE_LEVELS, runs, bins=60, xmax_percentile=99.5, isis=(1,17))

In [ ]:
from utls.io_utils import *

model_info = dict(
    model_name="DistanceMemoryModel",
    metric="loglikelihood",
    noise_mode="diffuse",
    encoder="texture_statistics",
    rate=0.01,
    run_id="test"
)

save_dir = make_model_save_dir(save_base, model_info)
save_runs_summary(runs, model_info, save_dir)

In [ ]:
# NOISE_LEVELS = np.linspace(0.1, 1, 4)

# curves, runs = rocs_across_noise(
#     NOISE_LEVELS,
#     runner=run_experiment_scores,
#     X0=X0, name_to_idx=name_to_idx, experiment_list=experiment_list,
#     metric="loglikelihood",
#     noise_mode="decay"#   # or "loglikelihood"
# )

# plot_noise_overlays(curves)
# # for nv in NOISE_LEVELS:
# #     if nv in runs:
# #         plot_isi_and_half_for_noise(nv, runs[nv], isis=(1,17))
# #         plot_histograms_for_noise(nv, runs[nv], bins=60)

# plot_across_noise(NOISE_LEVELS, runs, bins=60, xmax_percentile=99.5, isis=(1,17))


all_runs = load_all_runs(save_base)

In [ ]:
all_runs[-1]['info']['model_info']['model_name']

In [ ]:
plot_noise_overlays(curves)
# for nv in NOISE_LEVELS:
#     if nv in runs:
#         plot_isi_and_half_for_noise(nv, runs[nv], isis=(1,17))
#         plot_histograms_for_noise(nv, runs[nv], bins=60)

plot_across_noise(NOISE_LEVELS, runs, bins=60, xmax_percentile=99.5, isis=(1,17))

In [ ]:
# # Fix sigma0 here
# SIGMA0_FIXED = 0.5
# NOISE_LEVELS = np.linspace(1, 2.5, 5)  # values for sigma_enc

# curves, runs = rocs_across_noise(
#     NOISE_LEVELS,
#     runner=run_experiment_noisy,
#     X0=X0, name_to_idx=name_to_idx, experiment_list=experiment_list,
#     sigma0=SIGMA0_FIXED,    # fixed
#     metric="mahalanobis", # or "mahalanobis"
#     epochs=1
# )

# plot_noise_overlays(curves)
# # for nv in NOISE_LEVELS:
# #     if nv in runs:
# #         plot_isi_and_half_for_noise(nv, runs[nv], isis=(1,17))
# #         plot_histograms_for_noise(nv, runs[nv], bins=60)

# plot_across_noise(NOISE_LEVELS, runs, bins=60, xmax_percentile=99.5, isis=(1,17))

In [ ]:
# Fix sigma0 here
SIGMA0_FIXED = 0.5
NOISE_LEVELS = np.linspace(1, 2.5, 5)  # values for sigma_enc

curves, runs = rocs_across_noise(
    NOISE_LEVELS,
    runner=run_experiment_noisy,
    X0=X0, name_to_idx=name_to_idx, experiment_list=experiment_list,
    sigma0=SIGMA0_FIXED,    # fixed
    metric="logliki", # or "mahalanobis"
    epochs=1
)

# Per-noise plots
plot_noise_overlays(curves)
for nv in NOISE_LEVELS:
    if nv in runs:
        plot_isi_and_half_for_noise(nv, runs[nv], isis=(1,17))
        plot_histograms_for_noise(nv, runs[nv], bins=60)

# NEW: summary ROC plots
plot_summary_rocs(runs)